In [4]:
!nvidia-smi

Thu Jul 16 16:20:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
!pip install ninja

In [6]:
import torch
import torch.nn.functional as F
from torch.utils.cpp_extension import load_inline

In [15]:
cuda_source = """
#include <cuda_runtime.h>
#include <torch/extension.h>
#include <c10/cuda/CUDAException.h>
#include <cuda.h>
#include <stdio.h>

__global__ void coalesced_wb_sgemv_kernel(const float* __restrict__ mat, const float* __restrict__ vec, float* __restrict__ res, int M, int N) {
    extern __shared__ float smem[];

    int bid = blockIdx.x;
    if (bid >= M) return;

    int tid = threadIdx.x;

    float partial_sum = 0.f;

    for (int col = tid; col < N; col += blockDim.x) {
        partial_sum += mat[bid * N + col] * vec[col];
    }

    int lane = tid % warpSize;
    int warp_id = tid / warpSize;

    for (int i = warpSize / 2; i > 0; i /= 2) {
        partial_sum += __shfl_down_sync(0xffffffff, partial_sum, i);
    }

    if (lane == 0) {
        smem[warp_id] = partial_sum;
    }
    __syncthreads();

    int num_warps = (blockDim.x + warpSize - 1) / warpSize;
    if (tid < num_warps) {
        partial_sum = smem[tid];
    } else {
        partial_sum = 0.f;
    }

    if (warp_id == 0) {
        for (int i = 1; i > 0; i >>= 1) {
            partial_sum += __shfl_down_sync(0xffffffff, partial_sum, i);
        }
        if (tid == 0) {
            smem[0] = partial_sum;
        }
    }
    __syncthreads();

    if (tid == 0) {
        res[bid] = smem[0];
    }
}

torch::Tensor gemv_forward(torch::Tensor mat, torch::Tensor vec) {
    auto mat_c = mat.contiguous();
    auto vec_c = vec.contiguous();

    const int M = mat_c.size(0);
    const int N = mat_c.size(1);

    auto res = torch::empty({M}, mat_c.options());

    int NUM_THREADS = 64;
    int warp_size = 32;

    dim3 block_size(NUM_THREADS);
    dim3 grid_size(M);
    size_t shared_mem_size = ((block_size.x + warp_size - 1) / warp_size) * sizeof(float);

    coalesced_wb_sgemv_kernel<<<grid_size, block_size, shared_mem_size>>>(
        mat_c.data_ptr<float>(), vec_c.data_ptr<float>(), res.data_ptr<float>(), M, N);

    return res;
}
"""

In [16]:
cpp_source = "torch::Tensor gemv_forward(torch::Tensor mat, torch::Tensor vec);"

In [17]:
import os
build_dir = "/tmp/gelu_build"
os.makedirs(build_dir, exist_ok=True)

In [18]:
module = load_inline(
    name="gemv_coalesced_wb",
    cpp_sources=cpp_source,
    cuda_sources=cuda_source,
    functions=["gemv_forward"],
    verbose=True,
    build_directory=build_dir
)

In [19]:
M, N = 4096, 8192
mat = torch.randn(M, N, device="cuda", dtype=torch.float32)
vec = torch.randn(N, device="cuda", dtype=torch.float32)

res = module.gemv_forward(mat, vec)
ref = torch.mv(mat, vec)

print("shape:", tuple(res.shape))
print("max abs diff:", (res - ref).abs().max().item())
print("allclose:", torch.allclose(res, ref, atol=1e-3, rtol=1e-4))

shape: (4096,)
max abs diff: 0.0001220703125
allclose: True
